In [ ]:
from revert_decoder import decode_revert, build_error_index

idx = build_error_index(
    abi_files=['./MyContract.abi.json'],          # or
    metadata_files=['./metadata.json'],           # standard solc metadata with output.abi
)

dec = decode_revert('0x246121cc3162ce0d2bbbd34de729511d90cae522aca7683951bb146239818ffd')        # Error(string)
print(dec.kind, dec.summary)

# dec2 = decode_revert('0x4e487b71' + '...')       # Panic(uint256)
# print(dec2.details['code_hex'], dec2.details['meaning'])

# dec3 = decode_revert('0x12345678' + '...', idx)  # Custom error via ABI
# print(dec3.summary)


In [ ]:
# If not installed:
# %pip install requests python-solcx

import os, json, math, textwrap, re
import requests
from typing import Dict, Any, List, Optional, Tuple

RPC_URL = os.environ.get("ETH_RPC_URL", "https://ethereum-mainnet.core.chainstack.com/0a0fc938e8b4772db93176722c762a1b")  
CHAIN_ID = int(os.environ.get("CHAIN_ID", "1"))

# From your file/module already available:
from revert_decoder import decode_revert, build_error_index


In [ ]:
def rpc(method: str, params: list) -> Any:
    r = requests.post(RPC_URL, json={"jsonrpc":"2.0","id":1,"method":method,"params":params}, timeout=180)
    r.raise_for_status()
    j = r.json()
    if "error" in j:
        raise RuntimeError(f"{method} error: {j['error']}")
    return j["result"]

def rpc_safe(method: str, params: list) -> Dict[str, Any]:
    try:
        r = requests.post(RPC_URL, json={"jsonrpc":"2.0","id":1,"method":method,"params":params}, timeout=180)
        ok = r.status_code == 200
        j = r.json() if ok else {"http_status": r.status_code, "text": r.text}
        return {"ok": ok and "error" not in j, "status": r.status_code, "data": j}
    except Exception as e:
        return {"ok": False, "status": None, "data": {"exc": str(e)}}


def get_tx(txhash: str) -> Dict[str, Any]:
    return rpc("eth_getTransactionByHash", [txhash])

def get_receipt(txhash: str) -> Dict[str, Any]:
    return rpc("eth_getTransactionReceipt", [txhash])

def get_code(address: str, block: str) -> str:
    return rpc("eth_getCode", [address, block])

def to_block_tag(block_number: int) -> str:
    return hex(block_number)


In [ ]:
# Tiny tracer: returns only the *first* failing frame ({address, depth, pc, returndata})
JS_TRACER = r"""
{
  res: null,
  step: function (log, db) {
    var op = log.op.toString();
    if (op === "REVERT" || op === "INVALID") {
      if (this.res === null) {
        this.res = {
          pc: log.getPC(),
          depth: log.getDepth(),
          address: log.getContract().toString(),
          op: op
        };
      }
    }
  },
  fault: function (log, db) {
    if (this.res === null) {
      this.res = {
        pc: log.getPC(),
        depth: log.getDepth(),
        address: log.getContract().toString(),
        fault: log.getError()
      };
    }
  },
  result: function (ctx, db) {
    if (this.res) {
      try {
        var rd = ctx.getReturnData ? ctx.getReturnData() : "";
        if (rd && rd.length) {
          // Some clients return bytes; stringify to hex without 0x
          this.res.returndata = (typeof rd === "string" && rd.startsWith("0x")) ? rd : ("0x" + rd);
        }
      } catch (e) {}
    }
    return this.res;
  }
}
"""

def trace_tx_for_revert(txhash: str, timeout: str = "60s") -> Optional[Dict[str, Any]]:
    params = [txhash, {"tracer": JS_TRACER, "timeout": timeout}]
    try:
        res = rpc("debug_traceTransaction", params)
        return res  # may be None if trace disabled
    except Exception as e:
        raise


In [ ]:
def sourcify_fetch_metadata(chain_id: int, address: str) -> Optional[Dict[str, Any]]:
    addr = address.lower()
    base = f"https://repo.sourcify.dev/contracts"
    for tier in ("full_match", "partial_match"):
        url = f"{base}/{tier}/{chain_id}/{addr}/metadata.json"
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            return r.json()
    return None

def sourcify_fetch_sources(chain_id: int, address: str) -> Dict[str, str]:
    """
    Download all source files in the Sourcify directory alongside metadata.json.
    """
    addr = address.lower()
    base = f"https://repo.sourcify.dev/contracts"
    for tier in ("full_match", "partial_match"):
        list_url = f"{base}/{tier}/{chain_id}/{addr}/"
        idx = requests.get(list_url, timeout=60)
        if idx.status_code != 200:
            continue
        # Parse simple HTML directory listing to find files
        files = re.findall(r'href="([^"]+)"', idx.text)
        out = {}
        for fn in files:
            if fn.endswith(".sol") or "/" in fn:  # nested paths are shown with subdirs
                file_url = f"{list_url}{fn}"
                rr = requests.get(file_url, timeout=60)
                if rr.status_code == 200:
                    out[fn] = rr.text
        if out:
            return out
    return {}


In [ ]:
import solcx

def ensure_solc(version: str):
    installed = set(solcx.get_installed_solc_versions())
    if version not in installed:
        solcx.install_solc(version)

def solcx_compile_from_metadata(metadata: Dict[str, Any], sources: Dict[str, str]) -> Dict[str, Any]:
    """
    Recompile using the exact compiler version and settings from metadata to get:
      - output.contracts[<path>][<Contract>].evm.deployedBytecode.sourceMap
      - output.contracts[...].abi
      - output.sources (with id mapping)
    """
    comp_ver = metadata.get("compiler", {}).get("version")
    if not comp_ver:
        raise RuntimeError("metadata.compiler.version not found")

    settings = metadata.get("settings", {})
    # Settings from metadata are suitable for standard-json input
    # Prepare sources mapping for standard-json
    std_sources = {path: {"content": content} for path, content in sources.items()}

    ensure_solc(comp_ver)
    solcx.set_solc_version(comp_ver)

    std_json = {
        "language": "Solidity",
        "settings": {
            **settings,
            "outputSelection": {
                "*": {
                    "*": [
                        "abi",
                        "evm.deployedBytecode.sourceMap",
                        "evm.bytecode.object",
                        "evm.deployedBytecode.object",
                        "metadata",
                    ],
                    "": ["ast"]
                }
            }
        },
        "sources": std_sources,
    }

    output = solcx.compile_standard(std_json, allow_paths=".")
    return output


In [ ]:
OPCODES: Dict[int, str] = {}
# Basic opcode table
_names = {
    0x00:"STOP",0x01:"ADD",0x02:"MUL",0x03:"SUB",0x04:"DIV",0x05:"SDIV",0x06:"MOD",0x07:"SMOD",
    0x08:"ADDMOD",0x09:"MULMOD",0x0a:"EXP",0x0b:"SIGNEXTEND",
    0x10:"LT",0x11:"GT",0x12:"SLT",0x13:"SGT",0x14:"EQ",0x15:"ISZERO",0x16:"AND",0x17:"OR",
    0x18:"XOR",0x19:"NOT",0x1a:"BYTE",0x1b:"SHL",0x1c:"SHR",0x1d:"SAR",
    0x20:"KECCAK256",
    0x30:"ADDRESS",0x31:"BALANCE",0x32:"ORIGIN",0x33:"CALLER",0x34:"CALLVALUE",0x35:"CALLDATALOAD",
    0x36:"CALLDATASIZE",0x37:"CALLDATACOPY",0x38:"CODESIZE",0x39:"CODECOPY",0x3a:"GASPRICE",
    0x3b:"EXTCODESIZE",0x3c:"EXTCODECOPY",0x3d:"RETURNDATASIZE",0x3e:"RETURNDATACOPY",0x3f:"EXTCODEHASH",
    0x40:"BLOCKHASH",0x41:"COINBASE",0x42:"TIMESTAMP",0x43:"NUMBER",0x44:"DIFFICULTY",0x45:"GASLIMIT",
    0x46:"CHAINID",0x47:"SELFBALANCE",0x48:"BASEFEE",0x49:"BLOBHASH",0x4a:"BLOBBASEFEE",
    0x50:"POP",0x51:"MLOAD",0x52:"MSTORE",0x53:"MSTORE8",0x54:"SLOAD",0x55:"SSTORE",0x56:"JUMP",
    0x57:"JUMPI",0x58:"PC",0x59:"MSIZE",0x5a:"GAS",0x5b:"JUMPDEST",
    0xf0:"CREATE",0xf1:"CALL",0xf2:"CALLCODE",0xf3:"RETURN",0xf4:"DELEGATECALL",0xf5:"CREATE2",
    0xfa:"STATICCALL",0xfd:"REVERT",0xfe:"INVALID",0xff:"SELFDESTRUCT"
}
OPCODES.update(_names)
for i in range(1, 33):
    OPCODES[0x5f + i] = f"PUSH{i}"
for i in range(1, 17):
    OPCODES[0x7f + i] = f"DUP{i}"
for i in range(1, 17):
    OPCODES[0x8f + i] = f"SWAP{i}"
for i in range(1, 5):
    OPCODES[0xa0 + i] = f"LOG{i}"

def hex_to_bytes(h: str) -> bytes:
    h = h[2:] if h.startswith("0x") else h
    return bytes.fromhex(h)

def disassemble(bytecode_hex: str) -> List[Tuple[int, str, bytes]]:
    """
    Returns a list of (pc, opcode_name, push_data_bytes)
    """
    b = hex_to_bytes(bytecode_hex)
    out = []
    i = 0
    n = len(b)
    while i < n:
        op = b[i]
        name = OPCODES.get(op, f"OP_{op:02x}")
        if 0x60 <= op <= 0x7f:  # PUSH1..PUSH32
            push_len = op - 0x5f
            data = b[i+1:i+1+push_len]
            out.append((i, name, data))
            i += 1 + push_len
        else:
            out.append((i, name, b""))
            i += 1
    return out

def pc_to_instruction_index(bytecode_hex: str, pc: int) -> int:
    instrs = disassemble(bytecode_hex)
    pc_to_idx = {p:i for i,(p,_,__) in enumerate(instrs)}
    if pc in pc_to_idx:
        return pc_to_idx[pc]
    # If PC falls inside PUSH data (rare for REVERT), snap back to last instr
    prev = [p for p,_n,_d in instrs if p <= pc]
    if not prev:
        return 0
    last = max(prev)
    return pc_to_idx[last]


In [ ]:
from dataclasses import dataclass

@dataclass
class SrcMapEntry:
    s: int      # start byte offset in the file
    l: int      # length
    f: int      # file ID
    j: str      # jump type
    m: str      # modifier depth

def parse_srcmap(srcmap: str) -> List[SrcMapEntry]:
    """
    srcmap is a semicolon-separated series of entries like s:l:f:j:m with inheritance of previous fields.
    """
    parts = srcmap.split(";")
    out: List[SrcMapEntry] = []
    prev = SrcMapEntry(0, 0, -1, "", "")
    for p in parts:
        if not p:
            out.append(prev)
            continue
        fields = p.split(":")
        s = int(fields[0]) if len(fields) > 0 and fields[0] != "" else prev.s
        l = int(fields[1]) if len(fields) > 1 and fields[1] != "" else prev.l
        f = int(fields[2]) if len(fields) > 2 and fields[2] != "" else prev.f
        j = fields[3] if len(fields) > 3 and fields[3] != "" else prev.j
        m = fields[4] if len(fields) > 4 and fields[4] != "" else prev.m
        prev = SrcMapEntry(s, l, f, j, m)
        out.append(prev)
    return out

def byte_span_to_linecol(source_text: str, start: int, length: int) -> Tuple[Tuple[int,int], Tuple[int,int]]:
    """
    Convert byte offsets to (line,col) 1-based pairs for start/end.
    """
    # Treat as UTF-8
    data = source_text.encode("utf-8")
    end = start + max(0, length)
    start = min(start, len(data))
    end = min(end, len(data))
    pre = data[:start].decode("utf-8", errors="ignore")
    seg = data[start:end].decode("utf-8", errors="ignore")
    # compute lines/cols
    def linecol(s: str) -> Tuple[int,int]:
        lines = s.splitlines(keepends=True)
        if not lines:
            return (1,1)
        line_no = len(lines)
        col = len(lines[-1]) + 1
        return (line_no, col)
    start_line, start_col = linecol(pre)
    seg_lines = seg.splitlines(keepends=True)
    if seg_lines:
        end_line = start_line + len(seg_lines) - 1
        end_col = (len(seg_lines[-1]) + (1 if seg.endswith("\n") else 1))
    else:
        end_line, end_col = start_line, start_col
    return (start_line, start_col), (end_line, end_col)


In [ ]:
def classify_snippet(snippet: str) -> str:
    s = snippet.lower()
    if "require" in s:
        return "Require"
    if "revert" in s:
        return "Revert"
    if "assert" in s:
        return "Assert"
    return "Unknown"

def build_error_index_from_output(compiled_output: Dict[str, Any]) -> Any:
    """
    Collect all ABIs from the compiled output and build the error index for custom errors decoding.
    """
    abi_files = []
    abis = []
    for src_path, contracts in compiled_output.get("contracts", {}).items():
        for cname, cd in contracts.items():
            abi = cd.get("abi")
            if isinstance(abi, list):
                abis.append(abi)
    # Use build_error_index convenience by writing temp ABI arrays? We can add directly:
    from revert_decoder import ErrorIndex  # uses your module's class
    idx = ErrorIndex()
    for a in abis:
        idx.add_abi(a, source="solcx")
    return idx

def analyze_failed_tx(txhash: str) -> Dict[str, Any]:
    tx = get_tx(txhash)
    if not tx or not tx.get("blockNumber"):
        raise RuntimeError("Tx not found or pending.")
    receipt = get_receipt(txhash)
    block_tag = tx["blockNumber"]  # hex
    trace = trace_tx_for_revert(txhash)
    if not trace:
        raise RuntimeError("Trace returned empty. Is debug/trace enabled on the node?")

    fail_addr = trace.get("address")
    pc = trace.get("pc")
    rdata = trace.get("returndata", "0x")

    # 1) Fetch verified metadata + sources
    meta = sourcify_fetch_metadata(CHAIN_ID, fail_addr) or {}
    sources = sourcify_fetch_sources(CHAIN_ID, fail_addr)
    if not meta or not sources:
        # still proceed with decode only
        err = decode_revert(rdata)
        return {
            "tx": txhash,
            "address": fail_addr,
            "pc": pc,
            "revert": {"kind": err.kind, "summary": err.summary, "selector": err.selector, "details": err.details},
            "sourceMapping": {"status":"unavailable", "reason":"No Sourcify metadata/sources"}
        }

    # 2) Compile to get runtime sourcemap + ABI
    out = solcx_compile_from_metadata(meta, sources)

    # Find the *runtime* sourcemap for the contract that matches fail_addr bytecode
    # Heuristic: pick the contract whose deployedBytecode.object is a prefix of on-chain code
    onchain_code = get_code(fail_addr, block_tag)
    best = None
    best_len = -1
    best_key = None
    for src_path, contracts in out.get("contracts", {}).items():
        for cname, cd in contracts.items():
            db = cd.get("evm", {}).get("deployedBytecode", {})
            obj = db.get("object", "")
            smap = db.get("sourceMap", "")
            if not obj or not smap:
                continue
            if onchain_code.startswith("0x"+obj):
                if len(obj) > best_len:
                    best = (src_path, cname, cd)
                    best_len = len(obj)
                    best_key = (src_path, cname)
    if best is None:
        # Allow approximate (optimizer metadata / metadata hash differences). Fall back to the largest match by prefix.
        for src_path, contracts in out.get("contracts", {}).items():
            for cname, cd in contracts.items():
                db = cd.get("evm", {}).get("deployedBytecode", {})
                obj = db.get("object", "")
                smap = db.get("sourceMap", "")
                if not obj or not smap:
                    continue
                # Accept short prefix match (first N nybbles)
                N = 80  # 40 bytes
                if onchain_code[2:2+N] == obj[:N]:
                    best = (src_path, cname, cd); best_key=(src_path,cname); break
            if best: break

    # Build error index for custom errors
    err_index = build_error_index_from_output(out)

    dec = decode_revert(rdata, err_index)

    # If mapping failed, return decode-only
    if best is None:
        return {
            "tx": txhash,
            "address": fail_addr,
            "pc": pc,
            "revert": {"kind": dec.kind, "summary": dec.summary, "selector": dec.selector, "details": dec.details},
            "sourceMapping": {"status":"no-contract-match"}
        }

    src_path, cname, cd = best
    db = cd["evm"]["deployedBytecode"]
    obj = db["object"]
    srcmap = db["sourceMap"]
    instr_idx = pc_to_instruction_index(onchain_code, pc)
    entries = parse_srcmap(srcmap)

    # Bound-check: srcmap can be shorter; use last entry if needed
    entry = entries[instr_idx] if instr_idx < len(entries) else entries[-1]
    # Locate file id mapping: solc standard output has output.sources with ids
    # Build index: path -> id
    path_to_id = {}
    id_to_path = {}
    sources_out = out.get("sources", {})
    for i, (p, v) in enumerate(sources_out.items()):
        sid = v.get("id", i)
        path_to_id[p] = sid
        id_to_path[sid] = p
    file_id = entry.f
    file_path = id_to_path.get(file_id)
    if file_path is None:
        # fallback: best effort by ordering
        file_path = list(sources.keys())[file_id] if (0 <= file_id < len(sources)) else None

    snippet = None
    loc = None
    if file_path and file_path in sources:
        src_text = sources[file_path]
        s = entry.s
        l = entry.l
        snippet = src_text[s:s+l] if l > 0 else src_text[s:s+1]
        (sl, sc), (el, ec) = byte_span_to_linecol(src_text, s, l)
        loc = {"file": file_path, "start": {"line": sl, "col": sc}, "end": {"line": el, "col": ec}}

    return {
        "tx": txhash,
        "blockNumber": int(tx["blockNumber"], 16),
        "address": fail_addr,
        "depth": trace.get("depth"),
        "pc": pc,
        "opcode": trace.get("op"),
        "revert": {"kind": dec.kind, "summary": dec.summary, "selector": dec.selector, "details": dec.details},
        "sourceMapping": {
            "status": "ok" if loc else "partial",
            "location": loc,
            "classification": classify_snippet(snippet or ""),
            "snippet": snippet
        }
    }

def chainstack_trace_calltracer(txhash: str, timeout="60s"):
    params = [txhash, {"tracer": "callTracer", "timeout": timeout}]
    return rpc("debug_traceTransaction", params)  # requires debug_* (Geth or Erigon on Chainstack)


def chainstack_trace_replay(txhash: str):
    # Works when the node exposes trace_* (Erigon on Chainstack)
    return rpc("trace_replayTransaction", [txhash, ["trace","vmTrace"]])

def get_tx(txhash: str): return rpc("eth_getTransactionByHash", [txhash])

def eth_call_replay_revert_data(txhash: str) -> Optional[str]:
    tx = get_tx(txhash)
    if not tx or not tx.get("blockNumber"):
        return None
    call = {
        "from": tx.get("from"),
        "to": tx.get("to"),
        "gas": tx.get("gas") or hex(30_000_000),
        "gasPrice": tx.get("gasPrice"),
        "value": tx.get("value"),
        "data": tx.get("input"),
    }
    r = requests.post(RPC_URL, json={"jsonrpc":"2.0","id":1,"method":"eth_call","params":[call, tx["blockNumber"]]}, timeout=180)
    j = r.json()
    if isinstance(j.get("result"), str) and j["result"].startswith("0x"):
        return j["result"]
    err = (j.get("error") or {})
    if isinstance(err.get("data"), str) and err["data"].startswith("0x"):
        return err["data"]
    if isinstance(err.get("data"), dict) and isinstance(err["data"].get("data"), str) and err["data"]["data"].startswith("0x"):
        return err["data"]["data"]
    # scan nested objects (client-dependent)
    for v in (err.get("data") or {}).values() if isinstance(err.get("data"), dict) else []:
        if isinstance(v, dict) and isinstance(v.get("data"), str) and v["data"].startswith("0x"):
            return v["data"]
    m = re.search(r"0x[0-9a-fA-F]{8,}", json.dumps(err, ensure_ascii=False))
    return m.group(0) if m else None




In [ ]:
# Minimal custom tracer (single-line, single quotes inside):
CUSTOM_TRACER = "{res:null,step:function(log,db){var op=log.op.toString();if((op==='REVERT'||op==='INVALID')&&this.res===null){this.res={pc:log.getPC(),depth:log.getDepth(),address:log.getContract().toString(),op:op};}},fault:function(log,db){if(this.res===null){this.res={pc:log.getPC(),depth:log.getDepth(),address:log.getContract().toString(),fault:log.getError()};}},result:function(ctx,db){if(this.res){try{var rd=ctx.getReturnData?ctx.getReturnData():'';if(rd&&rd.length){this.res.returndata=(typeof rd==='string' && rd.startsWith('0x'))?rd:('0x'+rd);}}catch(e){};}return this.res;}}"

def chainstack_trace_custom(txhash: str, timeout="60s"):
    params = [txhash, {"tracer": CUSTOM_TRACER, "timeout": timeout}]
    out = rpc("debug_traceTransaction", params)   # requires debug_*; custom tracers require Enterprise dedicated
    return out


In [ ]:


from revert_decoder import decode_revert  # you already have this

def chainstack_analyze(txhash: str):
    # 1) try custom JS tracer (Enterprise dedicated)
    try:
        tr = chainstack_trace_custom(txhash)
        if tr:
            dec = decode_revert(tr.get("returndata","0x") or "")
            return {"mode":"customTracer", "address": tr.get("address"), "pc": tr.get("pc"),
                    "returndata": tr.get("returndata"), "decoded": {"kind": dec.kind, "summary": dec.summary}}
    except Exception:
        pass

    # 2) try callTracer
    try:
        ct = chainstack_trace_calltracer(txhash)
        # deepest error frame (if any)
        def walk(n): 
            out=[n]; 
            [out.extend(walk(c)) for c in n.get("calls",[]) or []]; 
            return out
        flat = walk(ct) if isinstance(ct, dict) else []
        deepest = next((n for n in reversed(flat) if n.get("error")), None)
        rhex = ct.get("revert") if isinstance(ct.get("revert"), str) else eth_call_replay_revert_data(txhash)
        dec = decode_revert(rhex or "")
        return {"mode":"callTracer", "address": (deepest or {}).get("to") or (deepest or {}).get("from"),
                "pc": None, "returndata": rhex, "decoded": {"kind": dec.kind, "summary": dec.summary}}
    except Exception:
        pass

    # 3) try Erigon trace
    try:
        pr = chainstack_trace_replay(txhash)
        vm = (pr or {}).get("vmTrace") or {}
        ops = vm.get("ops") or []
        last_pc = None
        for op in ops:
            if op.get("op") in ("REVERT","INVALID"):
                last_pc = op.get("pc")
        rhex = eth_call_replay_revert_data(txhash)
        dec = decode_revert(rhex or "")
        # best-effort address
        addr = None
        for t in (pr or {}).get("trace", []):
            if t.get("error"):
                addr = t.get("action",{}).get("to")
        return {"mode":"trace_replayTransaction", "address": addr, "pc": last_pc,
                "returndata": rhex, "decoded": {"kind": dec.kind, "summary": dec.summary}}
    except Exception:
        pass

    # 4) final fallback: just eth_call replay
    rhex = eth_call_replay_revert_data(txhash)
    dec = decode_revert(rhex or "")
    return {"mode":"eth_call", "address": get_tx(txhash).get("to"), "pc": None,
            "returndata": rhex, "decoded": {"kind": dec.kind, "summary": dec.summary}}


In [ ]:

def rpc_raw(method: str, params: list) -> requests.Response:
    return requests.post(RPC_URL, json={"jsonrpc":"2.0","id":1,"method":method,"params":params}, timeout=120)

def eth_call_replay_strict(txhash: str):
    tx = rpc("eth_getTransactionByHash", [txhash])
    call = {
        "from": tx.get("from"),
        **({} if tx.get("to") in (None, "0x0000000000000000000000000000000000000000") else {"to": tx.get("to")}),
        "gas": tx.get("gas") or hex(30_000_000),
        "maxFeePerGas": tx.get("maxFeePerGas"),           # harmless for call
        "maxPriorityFeePerGas": tx.get("maxPriorityFeePerGas"),
        "value": tx.get("value"),
        "data": tx.get("input"),
    }
    resp = rpc_raw("eth_call", [call, tx["blockNumber"]])
    j = resp.json()
    # Success path (rare for reverted tx)
    if isinstance(j.get("result"), str) and j["result"].startswith("0x"):
        return j["result"]
    err = (j.get("error") or {})
    # Common shapes: error.data = "0x.."
    if isinstance(err.get("data"), str) and err["data"].startswith("0x"): return err["data"]
    # error.data = { data: "0x.." } (geth/erigon variants)
    if isinstance(err.get("data"), dict):
        inner = err["data"].get("data")
        if isinstance(inner, str) and inner.startswith("0x"): return inner
        # nested per-tx object
        for v in err["data"].values():
            if isinstance(v, dict) and isinstance(v.get("data"), str) and v["data"].startswith("0x"):
                return v["data"]
    # last-ditch: scan message
    msg = (err.get("message") or "") + " " + json.dumps(err, ensure_ascii=False)
    m = re.search(r"0x[0-9a-fA-F]{8,}", msg)
    return m.group(0) if m else None

hexdata = eth_call_replay_strict(txh)
dec = decode_revert(hexdata or "")
{"mode":"eth_call(strict)", "decoded":{"kind": dec.kind, "summary": dec.summary}}


In [ ]:
txh = "0x246121cc3162ce0d2bbbd34de729511d90cae522aca7683951bb146239818ffd"
chainstack_analyze(txh)

In [ ]:
# Example 
txh = "0x246121cc3162ce0d2bbbd34de729511d90cae522aca7683951bb146239818ffd"
result = analyze_failed_tx(txh)
print(result)

## Fork and use Foundry for TX tracing

In [ ]:
from dotenv import load_dotenv

# === CONFIG ===
# Set a free/your own upstream RPC that allows forking (latest block is fine).
# Archive is ideal if you want to fork at an old block; otherwise we'll try latest.
load_dotenv()

UPSTREAM_URL = os.environ.get("ETH_FORK_RPC", "").strip() or "https://cloudflare-eth.com"

# Local anvil port (auto-pick if None)
ANVIL_PORT = None   # e.g., 8547 if you prefer a fixed port

# Tx you want to analyze
TXH = "0x246121cc3162ce0d2bbbd34de729511d90cae522aca7683951bb146239818ffd"


In [ ]:
import os, time, json, socket, subprocess, shlex, signal, requests
from contextlib import contextmanager
from typing import Any, Dict, Optional

def rpc(url: str, method: str, params: list, timeout=60) -> Any:
    r = requests.post(url, json={"jsonrpc":"2.0","id":1,"method":method,"params":params}, timeout=timeout)
    r.raise_for_status()
    j = r.json()
    if "error" in j:
        raise RuntimeError(f"{method} error: {j['error']}")
    return j["result"]

def try_rpc(url: str, method: str, params: list, timeout=15) -> Optional[Any]:
    try:
        return rpc(url, method, params, timeout=timeout)
    except Exception:
        return None

def free_tcp_port() -> int:
    s = socket.socket()
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port

def upstream_get_tx_and_block(txh: str) -> Dict[str, Any]:
    tx = rpc(UPSTREAM_URL, "eth_getTransactionByHash", [txh])
    if not tx or not tx.get("blockNumber"):
        raise RuntimeError("Transaction not found or pending on upstream.")
    return {"tx": tx, "block": int(tx["blockNumber"], 16)}


In [ ]:
@contextmanager
def run_anvil_fork(upstream_url: str, fork_block: Optional[int] = None, port: Optional[int] = None):
    if port is None:
        port = free_tcp_port()
    cmd = [
        "anvil",
        "--fork-url", upstream_url,
        "--port", str(port),
        "--chain-id", "1",
        "--silent",
        "--no-rate-limit",
    ]
    if fork_block is not None:
        cmd += ["--fork-block-number", str(fork_block)]
    # start
    try:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, preexec_fn=os.setsid)
    except FileNotFoundError:
        raise SystemExit("`anvil` not found. Install Foundry: curl -L https://foundry.paradigm.xyz | bash && foundryup")

    # wait until JSON-RPC responds
    url = f"http://127.0.0.1:{port}"
    for _ in range(120):
        if try_rpc(url, "web3_clientVersion", []) is not None:
            break
        time.sleep(0.5)
    else:
        try:
            out, err = proc.communicate(timeout=1)
        except Exception:
            out = err = b""
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        raise RuntimeError(f"Anvil failed to start. stdout={out.decode()} stderr={err.decode()}")

    try:
        yield url
    finally:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception:
            pass


In [ ]:
# Custom Geth-style JS tracer (single-line string).
# Captures {address, depth, pc, returndata} at the first failing frame.
CUSTOM_TRACER = "{res:null,step:function(l,d){var o=l.op.toString();if((o==='REVERT'||o==='INVALID')&&this.res===null){this.res={pc:l.getPC(),depth:l.getDepth(),address:l.getContract().toString(),op:o};}},fault:function(l,d){if(this.res===null){this.res={pc:l.getPC(),depth:l.getDepth(),address:l.getContract().toString(),fault:l.getError()};}},result:function(ctx,db){if(this.res){try{var rd=ctx.getReturnData?ctx.getReturnData():'';if(rd&&rd.length){this.res.returndata=(typeof rd==='string'&&rd.startsWith('0x'))?rd:('0x'+rd);}}catch(e){}}return this.res;}}"

def trace_with_custom(anvil_url: str, txh: str) -> Optional[Dict[str, Any]]:
    return try_rpc(anvil_url, "debug_traceTransaction", [txh, {"tracer": CUSTOM_TRACER, "timeout": "60s"}], timeout=120)

def trace_with_calltracer(anvil_url: str, txh: str) -> Optional[Dict[str, Any]]:
    return try_rpc(anvil_url, "debug_traceTransaction", [txh, {"tracer": "callTracer", "timeout": "60s"}], timeout=120)

def replay_eth_call(anvil_url: str, tx: Dict[str, Any]) -> Optional[str]:
    call = {
        "from": tx.get("from"),
        **({} if tx.get("to") in (None, "0x0000000000000000000000000000000000000000") else {"to": tx.get("to")}),
        "gas": tx.get("gas") or hex(30_000_000),
        "gasPrice": tx.get("gasPrice"),
        "maxFeePerGas": tx.get("maxFeePerGas"),
        "maxPriorityFeePerGas": tx.get("maxPriorityFeePerGas"),
        "value": tx.get("value"),
        "data": tx.get("input"),
    }
    # We want revert data → don't raise on error; parse error.data
    resp = requests.post(anvil_url, json={"jsonrpc":"2.0","id":1,"method":"eth_call","params":[call, "latest"]}, timeout=120)
    try:
        j = resp.json()
    except Exception:
        return None
    if isinstance(j.get("result"), str) and j["result"].startswith("0x"):
        return j["result"]
    err = j.get("error") or {}
    if isinstance(err.get("data"), str) and err["data"].startswith("0x"): return err["data"]
    if isinstance(err.get("data"), dict):
        inner = err["data"].get("data")
        if isinstance(inner, str) and inner.startswith("0x"): return inner
        for v in err["data"].values():
            if isinstance(v, dict) and isinstance(v.get("data"), str) and v["data"].startswith("0x"):
                return v["data"]
    return None


In [ ]:
from revert_decoder import decode_revert

ENABLE_SRCMAP = False  # set True if you’ve pasted the sourcemap/solcx cells earlier

def analyze_tx_via_local_fork(txh: str, prefer_block=True) -> Dict[str, Any]:
    up = upstream_get_tx_and_block(txh)
    tx, block = up["tx"], up["block"]

    # Try to fork at the tx's block; if upstream isn't archive, fall back to latest
    fork_blocks = [block] if prefer_block else []
    fork_blocks += [None]  # latest

    last_err = None
    for fb in fork_blocks:
        try:
            with run_anvil_fork(UPSTREAM_URL, fork_block=fb, port=ANVIL_PORT) as anvil_url:
                # 1) custom tracer
                tr = trace_with_custom(anvil_url, txh)
                if isinstance(tr, dict) and (tr.get("returndata") or tr.get("op") or tr.get("fault")):
                    data = tr.get("returndata") or ""
                    dec = decode_revert(data)
                    out = {
                        "mode": "customTracer",
                        "fork_block": fb if fb is not None else "latest",
                        "address": tr.get("address"),
                        "pc": tr.get("pc"),
                        "decoded": {"kind": dec.kind, "summary": dec.summary, "selector": dec.selector, "details": dec.details},
                    }
                    # (Optional) source map mapping here if ENABLE_SRCMAP and pc is present
                    return out

                # 2) callTracer fallback
                ct = trace_with_calltracer(anvil_url, txh)
                if isinstance(ct, dict):
                    # find deepest error node
                    def walk(n):
                        out = [n]
                        for c in n.get("calls",[]) or []:
                            out.extend(walk(c))
                        return out
                    flat = walk(ct)
                    deepest = next((n for n in reversed(flat) if n.get("error") or n.get("revertReason")), None)
                    rhex = None
                    for key in ("revert","returnValue","output"):
                        v = (deepest or {}).get(key)
                        if isinstance(v, str) and v.startswith("0x") and len(v) >= 10:
                            rhex = v; break
                    if not rhex:
                        rhex = replay_eth_call(anvil_url, tx)  # grab revert bytes
                    dec = decode_revert(rhex or "")
                    return {
                        "mode": "callTracer",
                        "fork_block": fb if fb is not None else "latest",
                        "address": (deepest or {}).get("to") or (deepest or {}).get("from"),
                        "pc": None,
                        "decoded": {"kind": dec.kind, "summary": dec.summary, "selector": dec.selector, "details": dec.details},
                    }

                # 3) last resort: eth_call only
                rhex = replay_eth_call(anvil_url, tx)
                dec = decode_revert(rhex or "")
                return {
                    "mode": "eth_call",
                    "fork_block": fb if fb is not None else "latest",
                    "address": tx.get("to"),
                    "pc": None,
                    "decoded": {"kind": dec.kind, "summary": dec.summary, "selector": dec.selector, "details": dec.details},
                }
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Failed to fork/trace (try archive upstream or a different RPC). Last error: {last_err}")


In [ ]:
import subprocess, os, textwrap, sys

install_script = r"""
set -e
curl -L https://foundry.paradigm.xyz | bash
~/.foundry/bin/foundryup
echo "FOUND: $(~/.foundry/bin/anvil --version)"
"""
subprocess.run(["bash","-lc", install_script], check=True)


In [ ]:
import os, shutil, subprocess, time, socket, signal, requests
from contextlib import contextmanager

def _resolve_anvil_bin() -> str:
    # 1) env override
    p = os.environ.get("ANVIL_BIN")
    if p and os.path.exists(p): return p
    # 2) PATH
    p = shutil.which("anvil")
    if p: return p
    # 3) Foundry default location
    p = os.path.expanduser("~/.foundry/bin/anvil")
    if os.path.exists(p): return p
    raise SystemExit("`anvil` not found. Set ANVIL_BIN to its path or install Foundry (see cell A).")

def _free_port() -> int:
    s = socket.socket(); s.bind(("127.0.0.1",0))
    port = s.getsockname()[1]; s.close(); return port

@contextmanager
def run_anvil_fork(upstream_url: str, fork_block: Optional[int] = None, port: Optional[int] = None):
    anvil_bin = _resolve_anvil_bin()
    if port is None: port = _free_port()
    cmd = [
        anvil_bin,
        "--fork-url", upstream_url,
        "--port", str(port),
        "--chain-id", "1",
        "--silent",
        "--no-rate-limit",
    ]
    if fork_block is not None:
        cmd += ["--fork-block-number", str(fork_block)]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, preexec_fn=os.setsid)
    url = f"http://127.0.0.1:{port}"
    # wait for JSON-RPC to respond
    for _ in range(120):
        try:
            r = requests.post(url, json={"jsonrpc":"2.0","id":1,"method":"web3_clientVersion","params":[]}, timeout=2)
            if r.ok: break
        except Exception: pass
        time.sleep(0.5)
    try:
        yield url
    finally:
        try: os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception: pass


In [ ]:
res = analyze_tx_via_local_fork(TXH)

In [ ]:
res

In [ ]:
# Uses the same helpers you already have: run_anvil_fork, get_upstream_tx, build_call_from_tx, trace_with_calltracer, rpc
import requests


import json, requests

UPSTREAM_URL = os.environ.get("ETH_FORK_RPC", "").strip() or "https://cloudflare-eth.com"
ANVIL_PORT = None   # or set a fixed port like 8547

def rpc(url: str, method: str, params: list, timeout=60):
    r = requests.post(url, json={"jsonrpc":"2.0","id":1,"method":method,"params":params}, timeout=timeout)
    r.raise_for_status()
    j = r.json()
    if "error" in j:
        raise RuntimeError(f"{method} error: {j['error']}")
    return j["result"]

def get_upstream_tx(txh: str):
    r = requests.post(UPSTREAM_URL, json={"jsonrpc":"2.0","id":1,"method":"eth_getTransactionByHash","params":[txh]}, timeout=60)
    r.raise_for_status()
    j = r.json()
    if "error" in j or not j.get("result"):
        raise RuntimeError(f"Upstream tx lookup failed: {j}")
    return j["result"]

def build_call_from_tx(tx: dict) -> dict:
    call = {
        "from": tx.get("from"),
        "gas": tx.get("gas") or hex(30_000_000),
        "gasPrice": tx.get("gasPrice"),
        "maxFeePerGas": tx.get("maxFeePerGas"),
        "maxPriorityFeePerGas": tx.get("maxPriorityFeePerGas"),
        "value": tx.get("value"),
        "data": tx.get("input"),
    }
    to = tx.get("to")
    if to and to != "0x0000000000000000000000000000000000000000":
        call["to"] = to
    return call

# Custom JS tracer as a single-line string (Geth-style, supported by Anvil)
CUSTOM_TRACER = "{res:null,step:function(l,d){var o=l.op.toString();if((o==='REVERT'||o==='INVALID')&&this.res===null){this.res={pc:l.getPC(),depth:l.getDepth(),address:l.getContract().toString(),op:o};}},fault:function(l,d){if(this.res===null){this.res={pc:l.getPC(),depth:l.getDepth(),address:l.getContract().toString(),fault:l.getError()};}},result:function(ctx,db){if(this.res){try{var rd=ctx.getReturnData?ctx.getReturnData():'';if(rd&&rd.length){this.res.returndata=(typeof rd==='string'&&rd.startsWith('0x'))?rd:('0x'+rd);}}catch(e){}}return this.res;}}"

def trace_with_debug_traceCall(anvil_url: str, call: dict, block: str = "latest"):
    return rpc(anvil_url, "debug_traceCall", [call, block, {"tracer": CUSTOM_TRACER, "timeout":"60s"}])


def trace_for_pc_and_address(txh: str, upstream_url: str):
    tx = get_upstream_tx(txh)
    call = build_call_from_tx(tx)
    block_hex = tx.get("blockNumber")
    fork_blocks = [int(block_hex,16)] if block_hex else []
    fork_blocks += [None]

    last_err = None
    for fb in fork_blocks:
        try:
            with run_anvil_fork(upstream_url, fork_block=fb, port=None) as anvil_url:
                # 1) failing address via callTracer (deepest error frame)
                ct = trace_with_calltracer(anvil_url, txh) or {}
                def walk(n):
                    out=[n]
                    for c in n.get("calls",[]) or []: out.extend(walk(c))
                    return out
                flat = walk(ct) if isinstance(ct, dict) else []
                deepest = next((n for n in reversed(flat) if n.get("error") or n.get("revertReason")), None)
                fail_addr = (deepest or {}).get("to") or (deepest or {}).get("from")

                # 2) structLogs to locate last REVERT/INVALID pc
                res = rpc(anvil_url, "debug_traceCall", [call, "latest", {
                    "timeout":"60s",
                    "disableStorage": True,
                    "disableStack": False,
                    "enableMemory": False,
                    "enableReturnData": True
                }])
                logs = res.get("structLogs") or []
                pc = None
                for log in logs:
                    if log.get("op") in ("REVERT","INVALID"):
                        pc = log.get("pc")
                # If no explicit REVERT step found (e.g., OOG), take last step
                if pc is None and logs:
                    pc = logs[-1].get("pc")

                # Revert bytes (if present in result or callTracer)
                rhex = res.get("returnValue")
                if not (isinstance(rhex, str) and rhex.startswith("0x") and len(rhex) >= 10):
                    rhex = ct.get("revert") if isinstance(ct.get("revert"), str) else None

                return {"anvil_url": anvil_url, "fork_block": fb if fb is not None else "latest",
                        "address": fail_addr, "pc": pc, "revert_hex": rhex}
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Could not obtain pc/address via local fork. Last error: {last_err}")

pc_info = trace_for_pc_and_address(TXH, UPSTREAM_URL)
pc_info


In [ ]:
from revert_decoder import decode_revert

def analyze_tx_via_local_fork(txh: str):
    # 1) pull tx (for calldata/from/value)
    tx = get_upstream_tx(txh)
    call = build_call_from_tx(tx)

    # try forking at the tx’s block if upstream supports historical; else latest
    block_hex = tx.get("blockNumber")
    fork_blocks = [int(block_hex,16)] if block_hex else []
    fork_blocks += [None]  # also try latest

    last_exc = None
    for fb in fork_blocks:
        try:
            with run_anvil_fork(UPSTREAM_URL, fork_block=fb, port=ANVIL_PORT) as anvil_url:
                # run custom tracer via debug_traceCall
                tr = trace_with_debug_traceCall(anvil_url, call, "latest")
                data = (tr or {}).get("returndata") or ""
                dec = decode_revert(data)
                return {
                    "mode": "debug_traceCall",
                    "fork_block": fb if fb is not None else "latest",
                    "address": (tr or {}).get("address"),
                    "pc": (tr or {}).get("pc"),
                    "decoded": {
                        "kind": dec.kind,
                        "summary": dec.summary,
                        "selector": dec.selector,
                        "details": dec.details
                    }
                }
        except Exception as e:
            last_exc = e
            continue
    raise RuntimeError(f"Could not fork/trace (try a different upstream RPC with historical state). Last error: {last_exc}")


In [ ]:
# %pip install requests python-solcx

import re, solcx, json
from typing import Dict, Any, List, Tuple

def sourcify_fetch_metadata(chain_id: int, address: str) -> Optional[Dict[str, Any]]:
    addr = address.lower()
    base = f"https://repo.sourcify.dev/contracts"
    for tier in ("full_match","partial_match"):
        url = f"{base}/{tier}/{chain_id}/{addr}/metadata.json"
        r = requests.get(url, timeout=60)
        if r.status_code == 200:
            return r.json()
    return None

def sourcify_fetch_sources(chain_id: int, address: str) -> Dict[str,str]:
    addr = address.lower()
    base = f"https://repo.sourcify.dev/contracts"
    for tier in ("full_match","partial_match"):
        list_url = f"{base}/{tier}/{chain_id}/{addr}/"
        idx = requests.get(list_url, timeout=60)
        if idx.status_code != 200: continue
        files = re.findall(r'href="([^"]+)"', idx.text)
        out={}
        for fn in files:
            if fn.endswith(".sol"):
                rr = requests.get(f"{list_url}{fn}", timeout=60)
                if rr.status_code==200: out[fn]=rr.text
        if out: return out
    return {}

def ensure_solc(version: str):
    if version not in set(solcx.get_installed_solc_versions()):
        solcx.install_solc(version)
    solcx.set_solc_version(version)

def compile_from_metadata(metadata: Dict[str,Any], sources: Dict[str,str]) -> Dict[str,Any]:
    ver = metadata.get("compiler",{}).get("version")
    if not ver: raise RuntimeError("metadata.compiler.version missing")
    ensure_solc(ver)
    std_sources = {p: {"content": c} for p,c in sources.items()}
    settings = metadata.get("settings", {})
    std_json = {
        "language": "Solidity",
        "sources": std_sources,
        "settings": {
            **settings,
            "outputSelection": {"*":{"*":["abi","evm.deployedBytecode.sourceMap","evm.deployedBytecode.object"], "": ["ast"]}}
        }
    }
    return solcx.compile_standard(std_json, allow_paths=".")

# EVM disassembler to convert pc → instruction index
OPCODES = {**{0x00:"STOP",0x01:"ADD",0x02:"MUL",0x03:"SUB",0x04:"DIV",0x05:"SDIV",0x06:"MOD",0x07:"SMOD",
              0x08:"ADDMOD",0x09:"MULMOD",0x0a:"EXP",0x0b:"SIGNEXTEND",0x10:"LT",0x11:"GT",0x12:"SLT",0x13:"SGT",
              0x14:"EQ",0x15:"ISZERO",0x16:"AND",0x17:"OR",0x18:"XOR",0x19:"NOT",0x1a:"BYTE",0x1b:"SHL",0x1c:"SHR",
              0x1d:"SAR",0x20:"KECCAK256",0x30:"ADDRESS",0x31:"BALANCE",0x32:"ORIGIN",0x33:"CALLER",0x34:"CALLVALUE",
              0x35:"CALLDATALOAD",0x36:"CALLDATASIZE",0x37:"CALLDATACOPY",0x38:"CODESIZE",0x39:"CODECOPY",0x3a:"GASPRICE",
              0x3b:"EXTCODESIZE",0x3c:"EXTCODECOPY",0x3d:"RETURNDATASIZE",0x3e:"RETURNDATACOPY",0x3f:"EXTCODEHASH",
              0x40:"BLOCKHASH",0x41:"COINBASE",0x42:"TIMESTAMP",0x43:"NUMBER",0x44:"DIFFICULTY",0x45:"GASLIMIT",
              0x46:"CHAINID",0x47:"SELFBALANCE",0x48:"BASEFEE",0x49:"BLOBHASH",0x4a:"BLOBBASEFEE",
              0x50:"POP",0x51:"MLOAD",0x52:"MSTORE",0x53:"MSTORE8",0x54:"SLOAD",0x55:"SSTORE",0x56:"JUMP",0x57:"JUMPI",
              0x58:"PC",0x59:"MSIZE",0x5a:"GAS",0x5b:"JUMPDEST",0xf0:"CREATE",0xf1:"CALL",0xf2:"CALLCODE",0xf3:"RETURN",
              0xf4:"DELEGATECALL",0xf5:"CREATE2",0xfa:"STATICCALL",0xfd:"REVERT",0xfe:"INVALID",0xff:"SELFDESTRUCT"}}
for i in range(1,33): OPCODES[0x5f+i]=f"PUSH{i}"

def disassemble(bytecode_hex: str):
    b = bytes.fromhex(bytecode_hex[2:] if bytecode_hex.startswith("0x") else bytecode_hex)
    out=[]; i=0
    while i<len(b):
        op=b[i]; name=OPCODES.get(op, f"OP_{op:02x}")
        if 0x60<=op<=0x7f:
            n=op-0x5f; out.append((i,name,b[i+1:i+1+n])); i+=1+n
        else:
            out.append((i,name,b"")); i+=1
    return out

def pc_to_instr_index(bytecode_hex: str, pc: int) -> int:
    instrs = disassemble(bytecode_hex)
    idx = {p:i for i,(p,_,__) in enumerate(instrs)}
    if pc in idx: return idx[pc]
    # if pc in the middle of PUSH data, snap back
    prev = [p for p,_,__ in instrs if p <= pc]
    return idx[max(prev)] if prev else 0

# srcmap parsing & line/col calc
from dataclasses import dataclass
@dataclass
class SrcMapEntry: s:int; l:int; f:int; j:str; m:str
def parse_srcmap(srcmap: str) -> List[SrcMapEntry]:
    out=[]; prev=SrcMapEntry(0,0,-1,"","")
    for part in srcmap.split(";"):
        if not part: out.append(prev); continue
        f = part.split(":")
        s = int(f[0]) if len(f)>0 and f[0]!="" else prev.s
        l = int(f[1]) if len(f)>1 and f[1]!="" else prev.l
        fi= int(f[2]) if len(f)>2 and f[2]!="" else prev.f
        j  = f[3] if len(f)>3 and f[3]!="" else prev.j
        m  = f[4] if len(f)>4 and f[4]!="" else prev.m
        prev = SrcMapEntry(s,l,fi,j,m); out.append(prev)
    return out

def byte_span_to_linecol(src: str, start: int, length: int) -> Tuple[Tuple[int,int], Tuple[int,int]]:
    data = src.encode("utf-8")
    start = min(start, len(data)); end = min(start+max(0,length), len(data))
    pre = data[:start].decode("utf-8", errors="ignore")
    seg = data[start:end].decode("utf-8", errors="ignore")
    def linecol(s: str): 
        lines=s.splitlines(keepends=True)
        return (len(lines), len(lines[-1])+1 if lines else 1)
    sl, sc = linecol(pre)
    seg_lines = seg.splitlines(keepends=True)
    if seg_lines:
        el = sl + len(seg_lines) - 1
        ec = len(seg_lines[-1]) + 1
    else:
        el, ec = sl, sc
    return (sl, sc), (el, ec)

def classify_snippet(snippet: str) -> str:
    s = snippet.lower()
    if "require" in s: return "Require"
    if "revert" in s:  return "Revert"
    if "assert" in s:  return "Assert"
    return "Unknown"

def map_pc_to_source(anvil_url: str, fail_addr: str, pc: int, chain_id: int = 1):
    if not (fail_addr and isinstance(pc,int)):
        return {"status":"no-pc-or-address"}
    # 1) on-chain runtime bytecode at fork
    onchain_code = rpc(anvil_url, "eth_getCode", [fail_addr, "latest"])
    # 2) metadata/sources
    md = sourcify_fetch_metadata(chain_id, fail_addr)
    srcs = sourcify_fetch_sources(chain_id, fail_addr)
    if not md or not srcs:
        return {"status":"no-sourcify"}
    # 3) compile to get deployed sourceMap + object
    out = compile_from_metadata(md, srcs)
    # 4) find best contract match
    best=None; best_len=-1; best_key=None
    for spath, contracts in out.get("contracts",{}).items():
        for cname, cd in contracts.items():
            db = cd.get("evm",{}).get("deployedBytecode",{})
            obj = db.get("object",""); smap = db.get("sourceMap","")
            if not obj or not smap: continue
            if onchain_code.startswith("0x"+obj):
                if len(obj)>best_len:
                    best=(spath,cname,cd); best_len=len(obj); best_key=(spath,cname)
    if not best:
        # loose match (first ~40 bytes)
        for spath, contracts in out.get("contracts",{}).items():
            for cname, cd in contracts.items():
                obj = cd.get("evm",{}).get("deployedBytecode",{}).get("object","")
                if obj and onchain_code[2:2+80]==obj[:80]:
                    best=(spath,cname,cd); break
            if best: break
    if not best:
        return {"status":"no-bytecode-match"}
    spath,cname,cd = best
    db = cd["evm"]["deployedBytecode"]
    srcmap = db["sourceMap"]
    instr_idx = pc_to_instr_index(onchain_code, pc)
    entries = parse_srcmap(srcmap)
    entry = entries[instr_idx] if instr_idx < len(entries) else entries[-1]
    # path <-> id mapping
    id_to_path={}
    for p,v in out.get("sources",{}).items():
        sid=v.get("id") if isinstance(v,dict) and "id" in v else None
        if sid is not None: id_to_path[sid]=p
    fpath = id_to_path.get(entry.f)
    snippet=None; loc=None
    if fpath and fpath in srcs:
        src=srcs[fpath]; s=entry.s; l=entry.l
        snippet = src[s:s+l] if l>0 else src[s:s+1]
        (sl,sc),(el,ec)=byte_span_to_linecol(src, s, l)
        loc={"file": fpath, "start":{"line":sl,"col":sc}, "end":{"line":el,"col":ec}}
    return {"status":"ok","contract": {"path": spath, "name": cname}, "location": loc,
            "classification": classify_snippet(snippet or ""), "snippet": snippet}


In [ ]:
from revert_decoder import decode_revert

# Use the pc/address we just collected
fail_addr = pc_info["address"]
pc_val    = pc_info["pc"]
anvil_url = pc_info["anvil_url"]

# Decode (you already did this; just for completeness)
dec = decode_revert(pc_info.get("revert_hex") or "")

mapped = map_pc_to_source(anvil_url, fail_addr, pc_val, chain_id=1)
{"fail_addr": fail_addr, "pc": pc_val, "decoded": {"kind": dec.kind, "summary": dec.summary}, "mapped": mapped}


In [52]:
def get_code_upstream(address: str, block_number_int: Optional[int]):
    block_tag = hex(block_number_int) if isinstance(block_number_int, int) else "latest"
    try:
        r = requests.post(
            UPSTREAM_URL,
            json={"jsonrpc":"2.0","id":1,"method":"eth_getCode","params":[address, block_tag]},
            timeout=60
        )
        r.raise_for_status()
        j = r.json()
        if "result" in j and isinstance(j["result"], str) and j["result"].startswith("0x"):
            return j["result"]
    except Exception:
        pass
    return None


In [53]:
def map_pc_to_source_with_code(code_hex: str, fail_addr: str, pc: int, chain_id: int = 1):
    if not (isinstance(pc, int) and code_hex and code_hex.startswith("0x")):
        return {"status":"no-pc-or-code"}

    # 1) Pull metadata & sources from Sourcify
    md  = sourcify_fetch_metadata(chain_id, fail_addr)
    src = sourcify_fetch_sources(chain_id, fail_addr)
    if not md or not src:
        return {"status":"no-sourcify"}

    # 2) Recompile with the exact compiler/settings to get runtime srcmap and objects
    out = compile_from_metadata(md, src)

    # 3) Find the contract whose deployed bytecode matches on-chain code
    best = None; best_len = -1
    for spath, contracts in out.get("contracts", {}).items():
        for cname, cd in contracts.items():
            db = cd.get("evm", {}).get("deployedBytecode", {})
            obj = db.get("object", ""); smap = db.get("sourceMap", "")
            if not obj or not smap:
                continue
            if code_hex.startswith("0x" + obj) and len(obj) > best_len:
                best = (spath, cname, cd, smap); best_len = len(obj)

    # Loose prefix match if exact didn’t hit (metadata hash differences, etc.)
    if best is None:
        for spath, contracts in out.get("contracts", {}).items():
            for cname, cd in contracts.items():
                db = cd.get("evm", {}).get("deployedBytecode", {})
                obj = db.get("object", ""); smap = db.get("sourceMap", "")
                if obj and code_hex[2:2+80] == obj[:80]:
                    best = (spath, cname, cd, smap); break
            if best: break

    if best is None:
        return {"status":"no-bytecode-match"}

    spath, cname, cd, srcmap = best

    # 4) pc → instruction index → sourcemap entry
    instr_idx = pc_to_instr_index(code_hex, pc)
    entries   = parse_srcmap(srcmap)
    entry     = entries[instr_idx] if instr_idx < len(entries) else entries[-1]

    # 5) map file id → path, slice snippet and compute line/col
    id_to_path = {}
    for p, v in out.get("sources", {}).items():
        if isinstance(v, dict) and "id" in v:
            id_to_path[v["id"]] = p
    fpath = id_to_path.get(entry.f)

    snippet = None; loc = None
    if fpath and fpath in src:
        s_txt = src[fpath]
        s_off, s_len = entry.s, entry.l
        snippet = s_txt[s_off:s_off+s_len] if s_len > 0 else s_txt[s_off:s_off+1]
        (sl, sc), (el, ec) = byte_span_to_linecol(s_txt, s_off, s_len)
        loc = {"file": fpath, "start":{"line":sl,"col":sc}, "end":{"line":el,"col":ec}}

    return {
        "status": "ok" if loc else "partial",
        "contract": {"path": spath, "name": cname},
        "location": loc,
        "classification": classify_snippet(snippet or ""),
        "snippet": snippet
    }


In [54]:
# original tx block (so we ask for code at the right height)
orig_tx   = get_upstream_tx(TXH)
blk_int   = int(orig_tx["blockNumber"], 16)

# get runtime bytecode at that block (fallback to latest if the upstream isn’t archive)
code_hex  = get_code_upstream(fail_addr, blk_int) or get_code_upstream(fail_addr, None)

mapped2   = map_pc_to_source_with_code(code_hex or "", fail_addr, pc_val, chain_id=1)
{
  "fail_addr": fail_addr,
  "pc": pc_val,
  "decoded": {"kind": dec.kind, "summary": dec.summary},
  "mapped": mapped2
}


{'fail_addr': '0xcf91b70017eabde82c9671e30e5502d312ea6eb2',
 'pc': 1329,
 'decoded': {'kind': 'empty', 'summary': 'Empty revert data'},
 'mapped': {'status': 'no-sourcify'}}

In [56]:
new_TXH = "0x47331fd359673f43fcba62b9b7f88527b78fdd82d8815ce8716a4a0b2b3e7e45"
orig_tx   = get_upstream_tx(new_TXH)
blk_int   = int(orig_tx["blockNumber"], 16)

# get runtime bytecode at that block (fallback to latest if the upstream isn’t archive)
code_hex  = get_code_upstream(fail_addr, blk_int) or get_code_upstream(fail_addr, None)

mapped2   = map_pc_to_source_with_code(code_hex or "", fail_addr, pc_val, chain_id=1)
{
  "fail_addr": fail_addr,
  "pc": pc_val,
  "decoded": {"kind": dec.kind, "summary": dec.summary},
  "mapped": mapped2
}


{'fail_addr': '0xcf91b70017eabde82c9671e30e5502d312ea6eb2',
 'pc': 1329,
 'decoded': {'kind': 'empty', 'summary': 'Empty revert data'},
 'mapped': {'status': 'no-sourcify'}}

In [64]:
# Set your Etherscan API key:
load_dotenv()  # if using a .env file locally
ETHERSCAN_API_KEY = os.environ.get("ETHERSCAN_API_KEY", "").strip()
if not ETHERSCAN_API_KEY:
    print("⚠️ Set ETHERSCAN_API_KEY env var for rate limits and reliability.")
# --- Etherscan V2 config ---
ETHERSCAN_API_KEY = os.environ.get("ETHERSCAN_API_KEY", "").strip()
ETHERSCAN_BASE = "https://api.etherscan.io/v2/api"
ETHERSCAN_CHAIN_ID = 1  # Ethereum mainnet

def etherscan_get_source_v2(address: str, chain_id: int = ETHERSCAN_CHAIN_ID) -> dict:
    params = {
        "chainid": chain_id,
        "module": "contract",
        "action": "getsourcecode",
        "address": address,
        "apikey": ETHERSCAN_API_KEY or "",
    }
    r = requests.get(ETHERSCAN_BASE, params=params, timeout=60)
    r.raise_for_status()
    j = r.json()
    if j.get("status") != "1" or not j.get("result"):
        raise RuntimeError(f"Etherscan V2 getsourcecode failed: {j}")
    return j["result"][0]

def _maybe_json(s: str):
    s = (s or "").strip()
    if not s:
        return None
    # Etherscan often returns JSON as a string (sometimes with extra braces)
    candidates = [s]
    if s.startswith("{") and s.endswith("}"):
        candidates.append(s[1:-1])
    for c in candidates:
        try:
            return json.loads(c)
        except Exception:
            pass
    return None

def parse_etherscan_standard_input(item: dict) -> tuple[dict, str]:
    """
    Build a standard-json input (sources/settings) + compiler version from Etherscan V2 result.
    Handles both multi-file JSON and flattened Solidity strings.
    """
    src_raw = item.get("SourceCode") or ""
    std = _maybe_json(src_raw)

    if std and isinstance(std, dict) and ("sources" in std or std.get("language") == "Solidity"):
        sources_in = std.get("sources") or {}
        sources = {}
        for p, v in sources_in.items():
            if isinstance(v, dict) and "content" in v:
                sources[p] = {"content": v["content"]}
            elif isinstance(v, str):
                sources[p] = {"content": v}
        settings = std.get("settings", {})
    else:
        # Flattened single-file
        code = src_raw
        name = item.get("ContractName") or "Contract"
        path = f"{name}.sol"
        sources = {path: {"content": code}}
        opt_used = (item.get("OptimizationUsed") == "1")
        runs = int(item.get("Runs") or 200)
        evm = item.get("EVMVersion") or "default"
        settings = {"optimizer": {"enabled": opt_used, "runs": runs}}
        if evm and evm != "default":
            settings["evmVersion"] = evm

    compver = (item.get("CompilerVersion") or "").lstrip("v")  # "v0.8.23" -> "0.8.23"

    std_input = {
        "language": "Solidity",
        "sources": sources,
        "settings": {
            **settings,
            "outputSelection": {
                "*": {"*": ["abi","evm.deployedBytecode.object","evm.deployedBytecode.sourceMap"], "": ["ast"]}
            }
        }
    }
    return std_input, compver

def resolve_proxy_implementation(item: dict) -> Optional[str]:
    # V2 still exposes Proxy/Implementation in the result
    if item.get("Proxy") == "1" and item.get("Implementation"):
        return item["Implementation"]
    return None


In [66]:
def ensure_solc(version: str):
    if version not in set(solcx.get_installed_solc_versions()):
        solcx.install_solc(version)
    solcx.set_solc_version(version)

def compile_from_etherscan(address: str) -> tuple[dict, dict, str]:
    """
    Returns (compiled_output, standard_input_sources, compiler_version).
    Follows proxy to its implementation automatically (Etherscan V2).
    """
    item = etherscan_get_source_v2(address)
    impl = resolve_proxy_implementation(item)
    src_item = etherscan_get_source_v2(impl) if impl else item

    std_input, compver = parse_etherscan_standard_input(src_item)
    ensure_solc(compver)
    output = solcx.compile_standard(std_input, allow_paths=".")
    return output, std_input["sources"], compver

def map_pc_to_source_with_etherscan(fail_addr: str, pc: int, block_int: Optional[int]) -> dict:
    # 1) runtime code at the tx block (fallback to latest if upstream isn’t archive)
    code_hex = get_code_upstream(fail_addr, block_int) or get_code_upstream(fail_addr, None)
    if not (code_hex and code_hex.startswith("0x")):
        return {"status": "no-code"}

    # 2) compile sources via Etherscan V2
    try:
        out, sources_map, compver = compile_from_etherscan(fail_addr)
    except Exception as e:
        return {"status": "etherscan-compile-failed", "error": str(e)}

    # 3) match deployed runtime bytecode
    best = None; best_len = -1
    for spath, contracts in out.get("contracts", {}).items():
        for cname, cd in contracts.items():
            db = cd.get("evm",{}).get("deployedBytecode",{})
            obj = db.get("object",""); smap = db.get("sourceMap","")
            if not obj or not smap:
                continue
            if code_hex.startswith("0x"+obj) and len(obj) > best_len:
                best = (spath, cname, cd, smap); best_len = len(obj)
    if best is None:
        # loose prefix match (~40 bytes)
        for spath, contracts in out.get("contracts", {}).items():
            for cname, cd in contracts.items():
                db = cd.get("evm",{}).get("deployedBytecode",{})
                obj = db.get("object",""); smap = db.get("sourceMap","")
                if obj and code_hex[2:2+80] == obj[:80]:
                    best = (spath, cname, cd, smap); break
            if best: break
    if best is None:
        return {"status":"no-bytecode-match"}

    spath, cname, cd, srcmap = best

    # 4) pc → instruction index → srcmap entry
    instr_idx = pc_to_instr_index(code_hex, pc)
    entries   = parse_srcmap(srcmap)
    entry     = entries[instr_idx] if instr_idx < len(entries) else entries[-1]

    # 5) file id → path + slice snippet
    id_to_path = {}
    for p, v in out.get("sources", {}).items():
        if isinstance(v, dict) and "id" in v:
            id_to_path[v["id"]] = p
    fpath = id_to_path.get(entry.f)

    snippet = None; loc = None
    if fpath and fpath in sources_map and isinstance(sources_map[fpath], dict):
        s_txt = sources_map[fpath].get("content")
        if s_txt is not None:
            s_off, s_len = entry.s, entry.l
            snippet = s_txt[s_off:s_off+s_len] if s_len > 0 else s_txt[s_off:s_off+1]
            (sl, sc), (el, ec) = byte_span_to_linecol(s_txt, s_off, s_len)
            loc = {"file": fpath, "start":{"line":sl,"col":sc}, "end":{"line":el,"col":ec}}

    return {
        "status": "ok" if loc else "partial",
        "contract": {"path": spath, "name": cname, "compiler": compver},
        "location": loc,
        "classification": classify_snippet(snippet or ""),
        "snippet": snippet
    }


In [67]:
orig_tx = get_upstream_tx(TXH)
blk_int = int(orig_tx["blockNumber"], 16)

mapped_es = map_pc_to_source_with_etherscan(fail_addr, pc_val, blk_int)
{
  "fail_addr": fail_addr,
  "pc": pc_val,
  "mapped": mapped_es
}


{'fail_addr': '0xcf91b70017eabde82c9671e30e5502d312ea6eb2',
 'pc': 1329,
 'mapped': {'status': 'etherscan-compile-failed',
  'error': 'Solc binary for v0.8.23+commit.f704f362 is not available for this OS'}}